# 03 — Scalability, baselines, and design attribution

This notebook uses identical candidates and immutable snapshots for
four methods: reject-only verification, Arbiter active-row repair,
full cyclic projection, and OSQP/QP projection within solver tolerance.
It reports cached gate time and snapshot-build-plus-gate time
separately, retains every raw trial, and adds relation-family,
active-budget, and incremental-update ablations. SQL parsing and
relation compilation are reported separately by Notebook 00; the
snapshot-build column must not be called full end-to-end construction.

Larger asset counts are deterministic SQL-derived workload states, not
synthetic random vectors.


In [1]:
import subprocess
import sys

# Install only dependencies that are absent or version-sensitive in Colab.
# The final environment manifest records every resolved version.
packages = [
    "duckdb==1.3.2",
    "osqp==1.0.4",
    "sqlglot==27.0.0",
    "pyarrow==19.0.1",
]
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "--quiet", *packages]
)
print("Required packages are available.")


Required packages are available.


In [2]:
from pathlib import Path
import hashlib
import importlib
import json
import os
import sys

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

USE_GOOGLE_DRIVE = True
if IN_COLAB and USE_GOOGLE_DRIVE:
    drive.mount("/content/drive")
    default_root = "/content/drive/MyDrive/Arbiter_ICDE_2027"
else:
    default_root = "/content/Arbiter_ICDE_2027"

PROJECT_ROOT = Path(os.environ.get("ARBITER_PROJECT_ROOT", default_root))
active_path = PROJECT_ROOT / "active_run.json"
if not active_path.exists():
    raise FileNotFoundError(
        "Run Notebook 00 first. It creates active_run.json and the shared code."
    )
active = json.loads(active_path.read_text(encoding="utf-8"))
RUN_MODE = active["run_mode"]
if active.get("suite_version") != "2026-08-01-audited-v4":
    raise RuntimeError(
        "The active run was created by an older suite. Run the audited "
        "Notebook 00 before continuing."
    )
RUN_DIR = PROJECT_ROOT / active["run_directory"]
CODE_DIR = PROJECT_ROOT / "code"
if not CODE_DIR.exists():
    raise FileNotFoundError("Shared code is missing. Rerun Notebook 00.")
manifest_path = CODE_DIR / "code_manifest.json"
if not manifest_path.exists():
    raise FileNotFoundError("Shared-code manifest is missing. Rerun Notebook 00.")
expected_hashes = json.loads(manifest_path.read_text(encoding="utf-8"))
for name, expected_hash in expected_hashes.items():
    module_path = CODE_DIR / name
    if not module_path.is_file():
        raise FileNotFoundError(f"Shared module is missing: {name}")
    observed_hash = hashlib.sha256(module_path.read_bytes()).hexdigest()
    if observed_hash != expected_hash:
        raise RuntimeError(f"Shared module hash mismatch: {name}")
code_dir_text = str(CODE_DIR.resolve())
if code_dir_text not in sys.path:
    sys.path.insert(0, code_dir_text)
for module_name in [
    "arbiter_core",
    "arbiter_workload",
    "arbiter_experiments",
    "arbiter_data",
]:
    sys.modules.pop(module_name, None)
importlib.invalidate_caches()
config = json.loads((RUN_DIR / "run_config.json").read_text(encoding="utf-8"))
print(f"Active run: {RUN_MODE}")
print(f"Run directory: {RUN_DIR}")


Mounted at /content/drive
Active run: full
Run directory: /content/drive/MyDrive/Arbiter_ICDE_2027/runs/full_audited_v4


In [3]:
import pandas as pd
import numpy as np
import shutil

from arbiter_experiments import (
    relation_subset,
    run_incremental_benchmark,
    run_scale_benchmark,
    summarize_scale_trials,
)

workload_dir = RUN_DIR / "00_workload"
proposal_dir = RUN_DIR / "01_proposals_attacks"
output_dir = RUN_DIR / "03_scalability"
if output_dir.exists():
    shutil.rmtree(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

all_assets = pd.read_csv(workload_dir / "sql_assets.csv")
assets = all_assets[
    all_assets["expected_supported"]
    .astype(str)
    .str.lower()
    .eq("true")
].copy()
relations = pd.read_csv(workload_dir / "compiled_relations.csv")
policy = pd.read_csv(proposal_dir / "controlled_policy.csv")

repair_options = {
    "max_sweeps": config["repair_sweeps"],
    "active_row_budget": config["repair_active_row_budget"],
    "step": config["repair_step"],
    "osqp_max_iter": config["osqp_max_iter"],
}
scale_trials, build_trials = run_scale_benchmark(
    assets=assets,
    relations=relations,
    policy=policy,
    scales=config["scale_points"],
    repetitions=config["timing_repetitions"],
    warmups=config["timing_warmups"],
    attack_cases_per_relation_type=config[
        "scale_attack_cases_per_relation_type"
    ],
    methods=[
        "reject_only",
        "arbiter_active",
        "full_cyclic",
        "osqp_qp",
    ],
    attack_budget=0.25,
    tolerance=config["tolerance"],
    repair_options=repair_options,
)
scale_summary = summarize_scale_trials(scale_trials)
scale_trials.to_csv(
    output_dir / "scale_trials_raw.csv", index=False
)
build_trials.to_csv(
    output_dir / "constraint_build_trials.csv", index=False
)
scale_summary.to_csv(
    output_dir / "scale_summary.csv", index=False
)
display(scale_summary)


,dataset,scale,method,trials,attack_cases,timing_repetitions,runtime_p50_ms,runtime_p95_ms,runtime_p99_ms,snapshot_build_plus_gate_p50_ms,snapshot_build_plus_gate_p95_ms,verified_rate,fallback_rate,rejection_rate,unsafe_release_rate,mean_violations_before,mean_violations_after,max_residual_after,mean_l1_adjustment
0,NYC Taxi 2024-01,128,arbiter_active,120,12,10,0.177116,0.352829,4.022139,18.690101,34.114765,1.0,0.0,0.0,0.0,2.166667,0.000000,2.220446e-16,0.226671
1,NYC Taxi 2024-01,128,full_cyclic,120,12,10,1.036803,4.037836,6.910275,20.427155,34.960267,1.0,0.0,0.0,0.0,2.166667,0.000000,2.220446e-16,0.226673
2,NYC Taxi 2024-01,128,osqp_qp,120,12,10,2.106991,7.268223,11.421983,21.161376,36.065768,1.0,0.0,0.0,0.0,2.166667,0.000000,2.220446e-16,0.226671
3,NYC Taxi 2024-01,128,reject_only,120,12,10,0.046779,0.091571,0.102742,16.491353,33.987215,0.0,0.0,1.0,0.0,2.166667,2.166667,6.800000e-01,0.000000
4,NYC Taxi 2024-01,512,arbiter_active,120,12,10,0.359682,0.519067,1.217343,37.210857,43.217536,1.0,0.0,0.0,0.0,5.666667,0.000000,2.220446e-16,0.226670
5,NYC Taxi 2024-01,512,full_cyclic,120,12,10,9.387242,11.856306,13.737974,46.307091,52.228546,1.0,0.0,0.0,0.0,5.666667,0.000000,2.220446e-16,0.261301
6,NYC Taxi 2024-01,512,osqp_qp,120,12,10,6.709245,9.396589,11.104334,44.248840,50.224718,1.0,0.0,0.0,0.0,5.666667,0.000000,2.220446e-16,0.226670
7,NYC Taxi 2024-01,512,reject_only,120,12,10,0.116624,0.158546,0.174609,36.979715,43.000028,0.0,0.0,1.0,0.0,5.666667,5.666667,6.800000e-01,0.000000
8,NYC Taxi 2024-01,1024,arbiter_active,120,12,10,0.490959,0.589968,1.417457,46.158820,68.349873,1.0,0.0,0.0,0.0,6.833333,0.000000,2.220446e-16,0.226670
9,NYC Taxi 2024-01,1024,full_cyclic,120,12,10,22.648816,39.882758,46.364139,68.784757,92.095434,1.0,0.0,0.0,0.0,6.833333,0.000000,7.812500e-08,0.338956


In [4]:
# Paired comparisons use trial-aligned method times. Confidence
# intervals resample independent attack-case medians with a fixed
# analysis seed. Timing repetitions are not treated as new attacks.
pivot = scale_trials.pivot_table(
    index=[
        "dataset",
        "scale",
        "attack_case_id",
        "repetition",
    ],
    columns="method",
    values="method_runtime_ms",
    aggfunc="first",
).reset_index()
comparison_rows = []
rng = np.random.default_rng(2027)
for (dataset, scale), group in pivot.groupby(["dataset", "scale"]):
    for baseline in ["full_cyclic", "osqp_qp"]:
        valid = group.dropna(
            subset=["arbiter_active", baseline]
        ).copy()
        valid["ratio"] = (
            valid[baseline] / valid["arbiter_active"]
        )
        case_medians = (
            valid.groupby("attack_case_id")["ratio"]
            .median()
            .to_numpy()
        )
        if len(case_medians):
            bootstrap = np.median(
                rng.choice(
                    case_medians,
                    size=(10000, len(case_medians)),
                    replace=True,
                ),
                axis=1,
            )
            low, high = np.quantile(bootstrap, [0.025, 0.975])
            median = float(np.median(case_medians))
        else:
            low = high = median = np.nan
        comparison_rows.append(
            {
                "dataset": dataset,
                "scale": int(scale),
                "baseline": baseline,
                "paired_trials": len(valid),
                "attack_case_clusters": len(case_medians),
                "speedup_median": median,
                "speedup_ci95_low": float(low),
                "speedup_ci95_high": float(high),
                "direction": (
                    "baseline latency divided by Arbiter latency"
                ),
            }
        )
paired = pd.DataFrame(comparison_rows)
paired.to_csv(
    output_dir / "paired_method_comparisons.csv", index=False
)
display(paired)


,dataset,scale,baseline,paired_trials,attack_case_clusters,speedup_median,speedup_ci95_low,speedup_ci95_high,direction
0,NYC Taxi 2024-01,128,full_cyclic,120,12,6.499523,6.082652,6.986055,baseline latency divided by Arbiter latency
1,NYC Taxi 2024-01,128,osqp_qp,120,12,11.920615,11.727097,12.883004,baseline latency divided by Arbiter latency
2,NYC Taxi 2024-01,512,full_cyclic,120,12,24.666772,23.156753,26.085173,baseline latency divided by Arbiter latency
3,NYC Taxi 2024-01,512,osqp_qp,120,12,20.710857,18.414758,21.267333,baseline latency divided by Arbiter latency
4,NYC Taxi 2024-01,1024,full_cyclic,120,12,45.839652,40.826957,48.514774,baseline latency divided by Arbiter latency
5,NYC Taxi 2024-01,1024,osqp_qp,120,12,35.908230,33.826573,39.868084,baseline latency divided by Arbiter latency
6,NYC Taxi 2024-01,2048,full_cyclic,120,12,59.115306,57.286137,69.053840,baseline latency divided by Arbiter latency
7,NYC Taxi 2024-01,2048,osqp_qp,120,12,71.007848,56.308691,76.220371,baseline latency divided by Arbiter latency
8,NYC Taxi 2024-01,4096,full_cyclic,120,12,77.685742,73.925679,126.070655,baseline latency divided by Arbiter latency
9,NYC Taxi 2024-01,4096,osqp_qp,120,12,128.169379,119.420924,158.544130,baseline latency divided by Arbiter latency


In [5]:
# Design attribution at a common supported scale.
ablation_scale = min(
    2048, int(config["max_assets_per_dataset"])
)
ablation_specs = [
    (
        "decomposition_only_budget_12000",
        relations[
            relations["relation_type"] == "decomposition"
        ],
        12000,
    ),
    ("all_relations_budget_64", relations, 64),
    ("all_relations_budget_512", relations, 512),
    ("all_relations_budget_12000", relations, 12000),
]
ablation_frames = []
ablation_repetitions = (
    3 if config["run_mode"] == "pilot" else 10
)
for label, relation_frame, row_budget in ablation_specs:
    trials, _ = run_scale_benchmark(
        assets=assets,
        relations=relation_frame,
        policy=policy,
        scales=[ablation_scale],
        repetitions=ablation_repetitions,
        warmups=config["timing_warmups"],
        attack_cases_per_relation_type=config[
            "scale_attack_cases_per_relation_type"
        ],
        methods=["arbiter_active"],
        attack_budget=0.25,
        tolerance=config["tolerance"],
        repair_options={
            **repair_options,
            "active_row_budget": row_budget,
        },
    )
    trials.insert(0, "ablation", label)
    ablation_frames.append(trials)
ablation_trials = pd.concat(ablation_frames, ignore_index=True)
ablation_summary = (
    ablation_trials.groupby(
        ["dataset", "scale", "ablation"], as_index=False
    )
    .agg(
        trials=("method_runtime_ms", "size"),
        gate_p50_ms=("method_runtime_ms", "median"),
        gate_p95_ms=(
            "method_runtime_ms",
            lambda values: float(np.quantile(values, 0.95)),
        ),
        verified_rate=("verified", "mean"),
        fallback_rate=("fallback_used", "mean"),
        rejection_rate=("rejected", "mean"),
        l1_mean=("l1_adjustment", "mean"),
    )
)
ablation_trials.to_csv(
    output_dir / "ablation_trials_raw.csv", index=False
)
ablation_summary.to_csv(
    output_dir / "ablation_summary.csv", index=False
)
display(ablation_summary)


,dataset,scale,ablation,trials,gate_p50_ms,gate_p95_ms,verified_rate,fallback_rate,rejection_rate,l1_mean
0,NYC Taxi 2024-01,2048,all_relations_budget_12000,120,0.738492,1.069205,1.0,0.0,0.0,0.226670
1,NYC Taxi 2024-01,2048,all_relations_budget_512,120,0.411297,0.487594,1.0,0.0,0.0,0.226670
2,NYC Taxi 2024-01,2048,all_relations_budget_64,120,0.414760,0.502431,1.0,0.0,0.0,0.226670
3,NYC Taxi 2024-01,2048,decomposition_only_budget_12000,40,0.216057,0.232176,1.0,0.0,0.0,0.680000
4,TPC-H SF1,2048,all_relations_budget_12000,120,0.706230,0.942056,1.0,0.0,0.0,0.241395
5,TPC-H SF1,2048,all_relations_budget_512,120,0.414535,0.766516,1.0,0.0,0.0,0.241395
6,TPC-H SF1,2048,all_relations_budget_64,120,0.729262,1.080338,1.0,0.0,0.0,0.241395
7,TPC-H SF1,2048,decomposition_only_budget_12000,40,0.197510,0.218469,1.0,0.0,0.0,0.680000
8,UCI Adult,2048,all_relations_budget_12000,120,0.500473,0.785978,1.0,0.0,0.0,0.226679
9,UCI Adult,2048,all_relations_budget_512,120,0.465382,0.595848,1.0,0.0,0.0,0.226679


In [6]:
incremental_frames = []
for dataset, dataset_assets in assets.groupby("dataset", sort=False):
    max_scale = min(
        max(config["scale_points"]), len(dataset_assets)
    )
    subset_assets = dataset_assets.head(max_scale)
    ordered_ids = subset_assets["asset_id"].astype(str).tolist()
    subset_relations = relation_subset(
        relations[relations["dataset"] == dataset],
        ordered_ids,
    )
    frame = run_incremental_benchmark(
        asset_ids=ordered_ids,
        relations=subset_relations,
        repetitions=config["timing_repetitions"],
    )
    frame.insert(0, "dataset", dataset)
    incremental_frames.append(frame)
incremental = pd.concat(incremental_frames, ignore_index=True)
incremental.to_csv(
    output_dir / "incremental_update_trials.csv", index=False
)
display(
    incremental.groupby(
        ["dataset", "batch_relations"], as_index=False
    )
    .agg(
        incremental_p50_ms=("incremental_ms", "median"),
        rebuild_p50_ms=("full_rebuild_ms", "median"),
        speedup_median=("speedup", "median"),
    )
)


,dataset,batch_relations,incremental_p50_ms,rebuild_p50_ms,speedup_median
0,NYC Taxi 2024-01,1,113.455737,238.728976,2.095949
1,NYC Taxi 2024-01,8,110.060034,237.567928,2.129194
2,NYC Taxi 2024-01,32,110.695807,245.443131,2.178796
3,TPC-H SF1,1,110.256500,227.800370,2.106081
4,TPC-H SF1,8,106.921213,245.660310,2.183672
5,TPC-H SF1,32,107.801475,244.786977,2.142838
6,UCI Adult,1,119.410493,243.186044,2.106034
7,UCI Adult,8,168.069277,256.317796,2.127467
8,UCI Adult,32,115.993777,240.874479,2.100273


In [7]:
trial_snapshot_consistency = (
    scale_trials.groupby(
        [
            "dataset",
            "scale",
            "attack_case_id",
            "repetition",
        ]
    )
    .agg(
        digests=("constraint_digest", "nunique"),
        pre_counts=("violations_before", "nunique"),
        candidate_digests=("candidate_digest", "nunique"),
    )
    .reset_index()
)
timing_order_counts = (
    scale_trials.groupby(
        ["dataset", "scale", "method", "execution_order"]
    )
    .size()
    .rename("count")
    .reset_index()
)
timing_order_spread = (
    timing_order_counts.groupby(
        ["dataset", "scale", "method"]
    )["count"]
    .agg(lambda values: int(values.max() - values.min()))
)
expected_scale_cells = {
    (str(dataset), int(scale))
    for dataset in assets["dataset"].unique()
    for scale in config["scale_points"]
}
observed_scale_cells = set(
    scale_trials[["dataset", "scale"]]
    .drop_duplicates()
    .itertuples(index=False, name=None)
)
observed_build_cells = set(
    build_trials[["dataset", "scale"]]
    .drop_duplicates()
    .itertuples(index=False, name=None)
)
method_count_ok = bool(
    (
        scale_trials.groupby(
            [
                "dataset",
                "scale",
                "attack_case_id",
                "repetition",
            ]
        )["method"].nunique()
        == 4
    ).all()
)
balanced_case_counts = (
    scale_trials.drop_duplicates(
        ["dataset", "scale", "attack_case_id"]
    )
    .groupby(
        ["dataset", "scale", "target_constraint_type"]
    )["attack_case_id"]
    .nunique()
)
quality = pd.DataFrame(
    [
        {
            "check": "every configured dataset/scale cell is present",
            "passed": bool(
                observed_scale_cells == expected_scale_cells
                and observed_build_cells == expected_scale_cells
            ),
        },
        {
            "check": "each candidate/repetition contains all methods",
            "passed": method_count_ok,
        },
        {
            "check": "scale attacks are balanced across relation families",
            "passed": bool(
                len(balanced_case_counts)
                and (
                    balanced_case_counts
                    == config[
                        "scale_attack_cases_per_relation_type"
                    ]
                ).all()
            ),
        },
        {
            "check": "all compared methods use the same snapshot/candidate",
            "passed": bool(
                (
                    (trial_snapshot_consistency["digests"] == 1)
                    & (trial_snapshot_consistency["pre_counts"] == 1)
                    & (
                        trial_snapshot_consistency[
                            "candidate_digests"
                        ]
                        == 1
                    )
                ).all()
            ),
        },
        {
            "check": "method execution order is counterbalanced",
            "passed": bool((timing_order_spread <= 1).all()),
        },
        {
            "check": "benchmark candidates contain violations",
            "passed": bool(
                (scale_trials["violations_before"] > 0).all()
            ),
        },
        {
            "check": "no measured unsafe release",
            "passed": bool(
                scale_trials["unsafe_release"].sum() == 0
            ),
        },
        {
            "check": "Arbiter releases only verified vectors",
            "passed": bool(
                scale_trials.loc[
                    scale_trials["method"] == "arbiter_active",
                    "verified",
                ].all()
            ),
        },
        {
            "check": "projection baselines reach verified vectors",
            "passed": bool(
                scale_trials.loc[
                    scale_trials["method"].isin(
                        ["full_cyclic", "osqp_qp"]
                    ),
                    "verified",
                ].all()
            ),
        },
        {
            "check": "QP baseline uses OSQP rather than local fallback",
            "passed": bool(
                scale_trials.loc[
                    scale_trials["method"] == "osqp_qp",
                    "solver",
                ].eq("osqp").all()
            ),
        },
        {
            "check": (
                "incremental and full builds have identical "
                "constraint-row semantics"
            ),
            "passed": bool(
                incremental["same_shape"].all()
                and incremental["same_constraint_count"].all()
                and incremental[
                    "same_constraint_row_multiset"
                ].all()
            ),
        },
    ]
)
quality.to_csv(output_dir / "stage_quality_checks.csv", index=False)
display(quality)
if not quality["passed"].all():
    raise RuntimeError(
        "Notebook 03 failed a benchmark-integrity check."
    )
print("Notebook 03 PASS. Continue to Notebook 04.")


,check,passed
0,every configured dataset/scale cell is present,True
1,each candidate/repetition contains all methods,True
2,scale attacks are balanced across relation fam...,True
3,all compared methods use the same snapshot/can...,True
4,method execution order is counterbalanced,True
5,benchmark candidates contain violations,True
6,no measured unsafe release,True
7,Arbiter releases only verified vectors,True
8,projection baselines reach verified vectors,True
9,QP baseline uses OSQP rather than local fallback,True


Notebook 03 PASS. Continue to Notebook 04.
